# 📊 Khám phá dữ liệu tuyển dụng TopCV
Notebook khám phá dữ liệu cũ `data/raw/raw_data.csv` (schema gộp yêu cầu/quyền lợi). Đây chưa phải notebook KDD hoàn chỉnh.

## 1. Import thư viện & đọc dữ liệu

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "Crawler").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "raw_data.csv"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "legacy_title_filtered.csv"

# Hiển thị font tiếng Việt
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (12, 5)

# Hiển thị đầy đủ nội dung cột
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.width', 200)

# Đọc file CSV
df = pd.read_csv(RAW_PATH, encoding='utf-8-sig')
print(f'✅ Đã đọc được {len(df)} tin tuyển dụng')
print(f'📋 Các cột: {list(df.columns)}')

ModuleNotFoundError: No module named 'matplotlib'

## 2. Xem tổng quan dữ liệu

In [ ]:
# 5 dòng đầu
df.head()

In [ ]:
# Thông tin tổng quan: kiểu dữ liệu, số null
df.info()

In [ ]:
# Thống kê số lượng giá trị null theo từng cột
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(1)
summary = pd.DataFrame({'Null count': null_counts, 'Null %': null_pct})
print(summary)

## 3. Thống kê theo Địa điểm

In [ ]:
# Top 15 địa điểm tuyển dụng
if 'Dia_Diem' in df.columns:
    top_location = df['Dia_Diem'].dropna().str.split(',').explode().str.strip()
    top_location = top_location.value_counts().head(15)
    
    top_location.plot(kind='barh', color='steelblue')
    plt.title('Top 15 địa điểm tuyển dụng')
    plt.xlabel('Số tin')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    print(top_location)

## 4. Thống kê Mức lương

In [ ]:
# Phân loại mức lương
if 'Muc_Luong_Goc' in df.columns:
    luong = df['Muc_Luong_Goc'].dropna()
    total = len(df)
    
    # Phân nhóm
    thoathuan = luong.str.contains('Thỏa thuận|Thoả thuận|thoa thuan', case=False, na=False).sum()
    co_so = total - thoathuan - luong.isnull().sum()
    
    print(f'  Thỏa thuận     : {thoathuan} tin ({thoathuan/total*100:.1f}%)')
    print(f'  Có mức lương cụ thể: {co_so} tin ({co_so/total*100:.1f}%)')
    print()
    print('--- Mẫu mức lương ---')
    print(luong[~luong.str.contains('Thỏa thuận|Thoả thuận', case=False, na=False)].head(20).to_string())

## 5. Thống kê Kinh nghiệm yêu cầu

In [ ]:
if 'Kinh_Nghiem' in df.columns:
    kn_counts = df['Kinh_Nghiem'].dropna().value_counts().head(12)
    
    kn_counts.plot(kind='bar', color='coral', edgecolor='black')
    plt.title('Yêu cầu kinh nghiệm')
    plt.ylabel('Số tin')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()
    print(kn_counts)

## 6. Top công ty tuyển dụng nhiều nhất

In [ ]:
if 'Ten_Cong_Ty' in df.columns:
    top_cty = df['Ten_Cong_Ty'].dropna().value_counts().head(15)
    
    top_cty.plot(kind='barh', color='mediumseagreen')
    plt.title('Top 15 công ty tuyển dụng nhiều nhất')
    plt.xlabel('Số tin')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## 7. Tìm kiếm theo từ khóa trong Mô tả / Yêu cầu

In [ ]:
# Tìm tin có yêu cầu kỹ năng cụ thể (thay đổi keyword bên dưới)
KEYWORD = 'Python'

mask = (
    df['Mo_Ta_Cong_Viec'].fillna('').str.contains(KEYWORD, case=False) |
    df['Yeu_Cau_Quyen_Loi'].fillna('').str.contains(KEYWORD, case=False)
)
result = df[mask][['Tieu_De', 'Ten_Cong_Ty', 'Muc_Luong_Goc', 'Dia_Diem']]
print(f'🔍 Tìm thấy {len(result)} tin có yêu cầu "{KEYWORD}"')
result.head(10)

## 8. Xuất kết quả lọc ra file mới

In [ ]:
# Chỉ lọc thiếu tiêu đề; chưa phải dữ liệu sạch theo bước 2.1.
df_filtered = df.dropna(subset=['Tieu_De'])
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_filtered.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f'Đã lưu {len(df_filtered)} tin vào {OUTPUT_PATH}')